# Lab 10 -- Decision Trees

Compared to last week, this is a very simple lab <span style="font-size:20pt;">😃</span> You'll have fun programming!

You will implement the **Classification and Regression Tree (CART)** algorithm from scratch.

The lab is broken down into the following pieces:

* Regression Criterion
* Creating Splits
* Buiding a Tree
* Making a prediction


# Decision trees for Regression
## Exercise 1 -- Download and load the dataset

We will be using the usual Boston Housing dataset, which is available to download from ECLASS

* Download the file
* Read it and separate the target variable from the features.
* Make a 80/10/10 train/validation/test split

In [478]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [479]:
housing_names = ["CRIM", "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS", "RAD", "TAX", "PTRATIO", "B", "LSTAT", "MEDV"]
data = pd.read_table("housing.txt", names=housing_names, sep=r'\s+')

The target variable will be as usual `MEDV`. Use the rest as features.

In [480]:
X = data.iloc[:, :-1].values
y = data["MEDV"].values

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5)

## Exercise 2 -- Optimization Criterion

For regression, a simple criterion to optimize is to minimize the sum of squared errors for a given region. This is, for all datapoints in a region with size, we minimize:

$$\sum_{i=1}^N(y_i - \hat{y})^2$$

where $N$ is the number of datapoits in the region and $\hat{y}$ is the mean value of the region for the target variable. 

Implement such a function using the description below.

Please, don't use an existing implementation, refer to the [book](https://www.statlearning.com/s/ISLRSeventhPrinting.pdf), and if you need help, ask questions!

In [481]:
def regression_criterion(region):
    """
    Implements the sum of squared error criterion in a region
    
    Parameters
    ----------
    region : ndarray
        Array of shape (N,) containing the values of the target values 
        for N datapoints in the training set.
    
    Returns
    -------
    float
        The sum of squared error
        
    Note
    ----
    The error for an empty region should be infinity (use: float("inf"))
    This avoids creating empty regions
    """
    if region.size == 0: 
        return float("inf")
        
    return np.sum((region - np.mean(region))**2)

In [482]:
# test your code
rng = np.random.default_rng(0)
print(regression_criterion(rng.random(size=40)))
print(regression_criterion(np.ones(10)))
print(regression_criterion(np.zeros(10)))
print(regression_criterion(np.array([])))

3.6200679838629544
0.0
0.0
inf


## Exercise 3 -- Make a split

In [483]:
def split_region(region, feature_index, tau):
    """
    Given a region, splits it based on the feature indicated by
    `feature_index`, the region will be split in two, where
    one side will contain all points with the feature with values 
    lower than `tau`, and the other split will contain the 
    remaining datapoints.
    
    Parameters
    ----------
    region : array of size (n_samples, n_features)
        a partition of the dataset (or the full dataset) to be split
    feature_index : int
        the index of the feature (column of the region array) used to make this partition
    tau : float
        The threshold used to make this partition
        
    Return
    ------
    left_partition : array
        indices of the datapoints in `region` where feature < `tau`
    right_partition : array
        indices of the datapoints in `region` where feature >= `tau` 
    """
    left_partition = np.where(region[:, feature_index] < tau)[0]
    right_partition = np.where(region[:, feature_index] >= tau)[0]

    return left_partition, right_partition

## Exercise 4 -- Find the best split

The strategy is quite simple (as well as inefficient), but it helps to reinforce the concepts.
We are going to use a greedy, exhaustive algorithm to select splits, selecting the `feature_index` and the `tau` that minimizes the Regression Criterion

In [484]:
def get_split(X, y):
    """
    Given a dataset (full or partial), splits it on the feature of that minimizes the sum of squared error
    
    Parameters
    ----------
    X : array (n_samples, n_features)
        features 
    y : array (n_samples, )
        labels
    
    Returns
    -------
    decision : dictionary
        keys are:
        * 'feature_index' -> an integer that indicates the feature (column) of `X` on which the data is split
        * 'tau' -> the threshold used to make the split
        * 'left_region' -> array of indices where the `feature_index`th feature of X is lower than `tau`
        * 'right_region' -> indices not in `low_region`
    """
    decision = dict()
    best_error = np.inf
    for feature_index in range(X.shape[1]):
        mini = X[:, feature_index].min()
        maxi = X[:, feature_index].max()
        if mini == maxi:
            continue

        step = (maxi - mini) / X.shape[0]
        for tau in np.arange(mini, maxi, step):
            left_partition, right_partition = split_region(X, feature_index, tau)

            left_error = regression_criterion(y[left_partition])
            right_error = regression_criterion(y[right_partition])
            total_error = (len(left_partition)/len(y))*left_error + (len(right_partition)/len(y))*right_error

            if total_error < best_error:
                best_error = total_error
                decision["feature_index"] = feature_index
                decision["tau"] = tau
                decision["left_region"] = left_partition
                decision["right_region"] = right_partition

    return decision

## Exercise 5 -- Recursive Splitting

The test above is an example on how to find the root node of our decision tree. The algorithm now is a greedy search until we reach a stop criterion. To find the actual root node of our decision tree, you must provide the whole training set, not just a slice of 15 rows as the test above.

The trivial stopping criterion is to recursively grow the tree until each split contains a single point (perfect node purity). If we go that far, it normally means we are overfitting.

You will implement these criteria to stop the growth:

* A node is a leaf if:
    * It has less than `min_samples` datapoints
    * It is at the `max_depth` level from the root (each split creates a new level)
    * The criterion is `0`



In [485]:
def recursive_growth(node, min_samples, max_depth, current_depth, X, y):
    """
    Recursively grows a decision tree.
    
    Parameters
    ----------
    node : dictionary
        If the node is terminal, it contains only the "value" key, which determines the value to be used as a prediction.
        If the node is not terminal, the dictionary has the structure defined by `get_split`
    min_samples : int
        parameter for stopping criterion if a node has <= min_samples datapoints
    max_depth : int
        parameter for stopping criterion if a node belongs to this depth
    depth : int
        current distance from the root
    X : array (n_samples, n_features)
        features (full dataset)
    y : array (n_samples, )
        labels (full dataset)
    
    Notes
    -----
    To create a terminal node, a dictionary is created with a single "value" key, with a value that
    is the mean of the target variable
    
    'left' and 'right' keys are added to non-terminal nodes, which contain (possibly terminal) nodes 
    from higher levels of the tree:
    'left' corresponds to the 'left_region' key, and 'right' to the 'right_region' key
    """
    left_idx = node["left_region"]
    right_idx = node["right_region"]

    X_left, y_left = X[left_idx], y[left_idx]
    X_right, y_right = X[right_idx], y[right_idx]

    if current_depth >= max_depth:
        node["left"] = {"value": np.mean(y_left) if len(y_left) > 0 else 0.0}
        node["right"] = {"value": np.mean(y_right) if len(y_right) > 0 else 0.0}

        return node

    if len(y_left) <= min_samples:
        node["left"] = {"value": np.mean(y_left) if len(y_left) > 0 else 0.0}
    else:
        left_decision = get_split(X_left, y_left)
        left_decision["left_region"]  = left_idx[left_decision["left_region"]]
        left_decision["right_region"] = left_idx[left_decision["right_region"]]
        node["left"] = recursive_growth(left_decision, min_samples, max_depth, current_depth + 1, X, y)

    if len(y_right) <= min_samples:
        node["right"] = {"value": np.mean(y_right) if len(y_right) > 0 else 0.0}
    else:
        right_decision = get_split(X_right, y_right)
        right_decision["left_region"]  = right_idx[right_decision["left_region"]]
        right_decision["right_region"] = right_idx[right_decision["right_region"]]
        node["right"] = recursive_growth(right_decision, min_samples, max_depth, current_depth + 1, X, y)

    return node

decision = get_split(X_train, y_train)
print(decision["feature_index"], decision["tau"])
print(X_train[decision["left_region"]][:, decision["feature_index"]])
print(X_train[decision["right_region"]][:, decision["feature_index"]])

print(regression_criterion(y_train))
print(regression_criterion(y_train[decision["left_region"]]))
print(regression_criterion(y_train[decision["right_region"]]))

12 9.71356435643565
[7.19 2.47 6.05 5.64 3.95 5.12 3.01 9.16 8.05 6.43 4.56 9.1  7.37 5.21
 8.43 6.72 5.29 7.79 4.81 7.2  3.11 7.67 6.36 6.58 7.14 7.56 7.39 1.73
 5.9  7.79 9.08 3.16 8.67 4.84 7.43 5.33 8.94 5.03 7.18 5.33 4.97 9.52
 4.98 3.56 6.68 9.22 6.62 6.78 9.14 5.28 5.08 7.7  7.74 7.12 3.92 5.77
 8.1  7.54 4.73 8.93 6.36 3.76 7.83 6.53 5.91 9.62 4.74 5.89 9.69 5.49
 9.71 6.07 1.98 4.63 7.9  8.77 2.94 7.6  9.43 7.34 8.58 8.61 3.16 4.03
 5.99 9.45 3.81 3.53 3.73 9.67 9.04 9.25 5.1  6.92 4.85 5.49 6.27 9.28
 6.65 4.69 4.21 4.38 6.9  9.51 3.26 7.44 5.52 3.11 4.45 4.59 5.04 5.5
 9.55 8.01 6.15 3.13 2.97 8.05 9.42 6.72 5.98 9.64 6.56 7.79 7.01 3.76
 3.57 5.81 9.53 7.88 9.5  3.53 3.7  8.26 6.57 4.61 6.19 8.05 5.5  6.58
 6.73 7.26 3.59 9.59 5.29 5.68 6.12 3.32 4.54 4.7  8.65 4.86 5.39 8.2
 6.87 4.67 8.81 6.93 5.19 9.38 8.88 5.98 7.39 7.73 7.53 8.23 6.75 5.25
 7.51]
[10.27 10.45 12.03 14.67 11.69 22.98 24.56 12.01 14.36 10.15 17.93 18.85
 14.09 14.65 21.02 31.99 17.28 17.12 21.52 22.88 1

Below we provide code to visualise the generated tree!

In [486]:
def create_tree(node, depth, structure = {}):
    if "value" in node:
        if depth in structure:
            structure[depth].append(node["value"])
        else: 
            structure[depth] = [node["value"]]
        return

    child_depth = depth + 1

    if depth in structure:
        structure[depth].append((node["feature_index"], node["tau"]))
    else:
        structure[depth] = [(node["feature_index"], node["tau"])]

    if "left" in node:
        create_tree(node["left"], child_depth, structure)

    if "right" in node:
        create_tree(node["right"], child_depth, structure)

    return structure

def print_tree(node, depth):
    tree = create_tree(node, depth)

    for depth, nodes in tree.items():
        print(" ".join(str(n) for n in nodes))

In [487]:
root = recursive_growth(get_split(X_train, y_train), 10, 4, 0, X_train, y_train)

print_tree(root, 0)

(12, np.float64(9.71356435643565))
(5, np.float64(6.803573964497033)) (12, np.float64(16.34702127659574))
(7, np.float64(2.920385046728973)) (5, np.float64(7.446838709677421)) (12, np.float64(13.390322580645151)) (4, np.float64(0.6569459459459468))
(0, np.float64(4.880548947368422)) (5, np.float64(6.541590909090932)) (12, np.float64(5.438157894736846)) (10, np.float64(17.80000000000001)) (10, np.float64(17.875675675675694)) (7, np.float64(2.3969120000000004)) (10, np.float64(19.28510638297872)) (0, np.float64(10.294532812499998))
(9, np.float64(287.6)) 50.0 (12, np.float64(7.575932203389835)) (6, np.float64(34.91379310344827)) (11, np.float64(388.0921052631581)) (5, np.float64(7.083684210526315)) (10, np.float64(14.920000000000002)) 34.900000000000006 (5, np.float64(6.0730416666666684)) (9, np.float64(355.9000000000002)) (12, np.float64(14.317894736842105)) (9, np.float64(299.8064516129032)) (8, np.float64(3.1764705882352935)) (5, np.float64(5.813299999999996)) (12, np.float64(18.89437

## Exercise 6 -- Make a Prediction
Use the a node to predict the class of a compatible dataset

In [488]:
def predict_sample(node, sample):
    """
    Makes a prediction based on the decision tree defined by `node`
    
    Parameters
    ----------
    node : dictionary
        A node created one of the methods above
    sample : array of size (n_features,)
        a sample datapoint
    """
    if "value" in node:
        return node["value"]
    
    feature = node["feature_index"]
    thresholder = node["tau"]

    if sample[feature] < thresholder:
        return predict_sample(node["left"], sample)
    if sample[feature] >= thresholder:
        return predict_sample(node["right"], sample)
        
def predict(node, X):
    """
    Makes a prediction based on the decision tree defined by `node`
    
    Parameters
    ----------
    node : dictionary
        A node created one of the methods above
    X : array of size (n_samples, n_features)
        n_samples predictions will be made
    """
    y = np.zeros(X.shape[0])
    for index, sample in enumerate(X):
        y[index] = predict_sample(node, sample)

    return y

Now use the functions defined above to calculate the RMSE of the validation set. 
* Try first with `min_samples=20` and `max_depth=6` (for this values you should get a validation RMSE of ~8.8)

Then, experiment with different values for the stopping criteria.

In [489]:
# calculate root mean squared error with numpy
def root_mean_squared_error(y_true, y_pred):
    """
    Calculates the root mean squared error between two arrays
    
    Parameters
    ----------
    y_true : array of size (n_samples,)
        true labels
    y_pred : array of size (n_samples,)
        predicted labels
    """
    return np.sqrt(np.mean((y_true - y_pred)**2))

In [490]:
root = get_split(X_train, y_train)
min_samples = 20
max_depth = 6
recursive_growth(root, min_samples, max_depth, 1, X_train, y_train)
train_rmse = root_mean_squared_error(y_train, predict(root, X_train))
test_rmse = root_mean_squared_error(y_test, predict(root, X_test))
val_rmse = root_mean_squared_error(y_val, predict(root, X_val))

print(f'Train RMSE : {train_rmse}')
print(f'Test RMSE : {test_rmse}')
print(f'Validation RMSE : {val_rmse}')

Train RMSE : 3.629195544770435
Test RMSE : 4.02190011517735
Validation RMSE : 3.6204553078061634


## Just to see how much we can improve this naive decision tree!

In [491]:
from sklearn.linear_model import LinearRegression

In [492]:
reg = LinearRegression().fit(X_train, y_train)

In [493]:
reg_train_rmse = root_mean_squared_error(y_train, reg.predict(X_train))
reg_test_rmse = root_mean_squared_error(y_test, reg.predict(X_test))
print(f'Regression Train RMSE : {reg_train_rmse}')
print(f'Regression Test RMSE : {reg_test_rmse}')

Regression Train RMSE : 4.726434151780453
Regression Test RMSE : 4.33233226695155


# Decision trees for Classification
You will implement decision trees for classification using the Gini index as the splitting criterion. You’ll build the tree recursively, selecting splits that minimize Gini impurity and classifying samples based on majority class in each leaf. A good dataset to start with is the Iris dataset, which is small, well-labeled, and available directly via `sklearn.datasets.load_iris().`

In [494]:
# Load the Iris dataset
from sklearn.datasets import load_iris

data = load_iris()
X = data.data
y = data.target

We will focus only on binary classification today!

In [495]:
X = X[y != 2]
y = y[y != 2]

In [496]:
# split your data into training, validation and test sets!

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2)
X_val, X_test, y_val, y_test = train_test_split(X, y, test_size=0.5)

### Feel free to use the same code as for regression, but change the criterion and the prediction function. You can also implement a new one if you want to!

In [497]:
# you will test 3 values for min_samples_split: 2, 4, 6
# Remember that this sets the minimum number of samples required in a node to be eligible for splitting. 
# These values are good for small datasets like Iris, but you can try other values for larger datasets to not make the tree too deep.

# your code goes here

def gini_criterion(region):
    if region.size == 0:
        return float("inf")
    
    classes, counts = np.unique(region, return_counts=True)
    probs = counts / len(region)
    return np.sum(probs * (1 - probs))

def get_split(X, y):
    decision = dict()
    best_error = np.inf
    for feature_index in range(X.shape[1]):
        min = X[:, feature_index].min()
        max = X[:, feature_index].max()
        if min == max:
            continue
        step = (max - min) / X.shape[0]
        for tau in np.arange(min, max, step):
            left_partition, right_partition = split_region(X, feature_index, tau)

            left_error = gini_criterion(y[left_partition])
            right_error = gini_criterion(y[right_partition])
            total_error = (len(left_partition)/len(y))*left_error + (len(right_partition)/len(y))*right_error

            if total_error < best_error:
                best_error = total_error
                decision["feature_index"] = feature_index
                decision["tau"] = tau
                decision["left_region"] = left_partition
                decision["right_region"] = right_partition

    return decision

def recursive_growth(node, min_samples, max_depth, current_depth, X, y):
    left_idx = node["left_region"]
    right_idx = node["right_region"]

    X_left, y_left = X[left_idx], y[left_idx]
    X_right, y_right = X[right_idx], y[right_idx]

    if current_depth >= max_depth:
        values_l, counts_l = np.unique(y_left, return_counts=True)
        idx_predom_l = np.argmax(counts_l)
        node["left"] = {"value": y_left[idx_predom_l]}

        values_r, counts_r = np.unique(y_right, return_counts=True)
        idx_predom_r = np.argmax(counts_r)
        node["right"] = {"value": y_right[idx_predom_r]}

        return node

    if len(y_left) <= min_samples:
        values, counts = np.unique(y_left, return_counts=True)
        idx_predom = np.argmax(counts)
        node["left"] = {"value": y_left[idx_predom]}
    else:
        left_decision = get_split(X_left, y_left)
        node["left"] = recursive_growth(left_decision, min_samples, max_depth, current_depth + 1, X, y)

    if len(y_right) <= min_samples:
        values, counts = np.unique(y_right, return_counts=True)
        idx_predom = np.argmax(counts)
        node["right"] = {"value": y_right[idx_predom]}
    else:
        right_decision = get_split(X_right, y_right)
        node["right"] = recursive_growth(right_decision, min_samples, max_depth, current_depth + 1, X, y)

    return node

def predict_sample(node, sample):
    if "value" in node:
        return node["value"]
    
    feature = node["feature_index"]
    thresholder = node["tau"]

    if sample[feature] < thresholder:
        return predict_sample(node["left"], sample)
    if sample[feature] >= thresholder:
        return predict_sample(node["right"], sample)
        
def predict(node, X):
    y = np.zeros(X.shape[0])
    for index, sample in enumerate(X):
        y[index] = predict_sample(node, sample)

    return y

Use accuracy to find the best split. Don't import it from sklearn, calculate it yourself, it's a one-liner ;)

In [498]:
## Lastly, evaluate your model on the test set and print the accuracy score

# your code goes here
root = recursive_growth(get_split(X_train, y_train), min_samples=2, max_depth=4, current_depth=0, X=X_train, y=y_train)

y_pred = predict(root, X_test)
accuracy = np.mean(y_pred == y_test)
print(f"Accuracy: {accuracy}")

Accuracy: 0.96
